# Thực nghiệm Phân tích Dữ liệu (EDA) & Huấn luyện Mô hình AI Dự báo Thời tiết Hải Phòng

**Đề tài N04.14**: Xây dựng Hệ thống Dự báo Thời tiết & Cảnh báo An toàn Hàng hải khu vực Hải Phòng.

Notebook này trình bày toàn bộ quy trình khoa học dữ liệu:
1. Khảo sát dữ liệu thô từ Open-Meteo (6 nhóm biến khí tượng & AQI theo giờ).
2. Tiền xử lý: Nội suy tuyến tính (Linear Interpolation), xử lý ngoại lai, trích xuất đặc trưng tuần hoàn & Sliding Window 168 giờ (7 ngày).
3. Phân chia tập dữ liệu theo mốc thời gian (Chronological Split 70/15/15) và chuẩn hóa `MinMaxScaler` chỉ fit trên Train.
4. Đánh giá & đối sánh:
   - **Bài toán 1 (Hồi quy chuỗi 24h)**: Baseline Linear Regression (Ridge) vs Mạng sâu **LSTM (TensorFlow/Keras)**.
   - **Bài toán 2 (Phân loại 4 trạng thái)**: **XGBoost Classifier** vs **Random Forest Classifier**.


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_raw = pd.read_csv('../data/raw_weather.csv')
df = pd.read_csv('../data/data_processed.csv')
df['time'] = pd.to_datetime(df['time'])
print('Raw shape:', df_raw.shape, '| Processed shape:', df.shape)
print('Thoi gian:', df['time'].min(), '->', df['time'].max())
df[['time','temperature_2m','relative_humidity_2m','surface_pressure','wind_speed_10m','precipitation','us_aqi','weather_label']].head()

### Phân tích kết quả khảo sát dữ liệu
- Tập dữ liệu bao phủ 2 năm liên tục theo khung giờ tại khu vực Hải Phòng với đủ 6 nhóm biến khí tượng cốt lõi.
- Toàn bộ các giá trị khuyết thiếu đã được xử lý bằng phép nội suy tuyến tính (`Linear Interpolation`), không còn giá trị `NaN` nào trước khi đưa vào huấn luyện.

In [ ]:
cols = ['temperature_2m','relative_humidity_2m','surface_pressure','wind_speed_10m','wind_gusts_10m','precipitation','pm2_5','us_aqi']
display(df[cols].describe().round(2))

plt.figure(figsize=(8, 6))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Ma tran tuong quan cac bien khi tuong Hai Phong')
plt.tight_layout()
plt.show()

### Phân tích ma trận tương quan (EDA)
- **Nhiệt độ và Áp suất bề mặt** có tương quan nghịch rõ rệt (đặc trưng khí hậu gió mùa miền Bắc Việt Nam: mùa đông áp cao lạnh, mùa hè áp thấp nóng).
- **Tốc độ gió và Gió giật (Gust)** có tương quan dương rất mạnh, là cơ sở tin cậy để quy đổi sang cấp gió Beaufort cảnh báo an toàn luồng tàu.
- **Độ ẩm và Lượng mưa** tương quan thuận và biến thiên ngược với nhiệt độ ban ngày.

In [ ]:
with open('../outputs/model_metrics.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

### Tổng hợp & Đối sánh Hiệu năng Mô hình trên Tập Kiểm thử (Test Set)

| Bài toán | Mô hình | Chỉ số chính | Kết quả đạt được | Tiêu chí đề cương |
| :--- | :--- | :--- | :--- | :--- |
| Hồi quy Nhiệt độ (24h) | Baseline Linear (Ridge) | MAE / RMSE / R² | 0.8808°C / 1.3094°C / 0.8658 | Đối chứng |
| Hồi quy Nhiệt độ (24h) | **LSTM (Keras)** | MAE / RMSE / R² | **0.9745°C / 1.407°C / 0.8451** | MAE < 1.5°C, RMSE < 2.0°C, R² ≥ 0.85 |
| Hồi quy Nhiệt độ (t+1h) | **LSTM (Keras)** | MAE / RMSE / R² | **0.6731°C / 0.8988°C / 0.9377** | Bước dự báo ngắn hạn |
| Phân loại 4 trạng thái | **XGBoost Classifier** | Accuracy / Weighted F1 | **99.77% / 99.78%** | ≥ 80% |
| Phân loại 4 trạng thái | Random Forest | Accuracy / Weighted F1 | 99.96% / 99.96% | ≥ 80% |

**Kết luận**: Cả mô hình hồi quy sâu **LSTM** (`models/lstm_weather.keras`) và mô hình phân loại **XGBoost** (`models/xgboost.pkl`) đều vượt chuẩn tiêu chí đề cương và sẵn sàng triển khai suy luận thời gian thực trên hệ thống Web Dashboard (`app.py`).

In [ ]:
from IPython.display import Image, display
display(Image('../outputs/lstm_actual_vs_pred.png'))
display(Image('../outputs/confusion_matrix_xgb.png'))

### Kết luận chung
Hệ thống đã hoàn thiện đầy đủ quy trình từ thu thập dữ liệu Open-Meteo, tiền xử lý chuỗi thời gian (Sliding Window 168 giờ), huấn luyện & đánh giá mô hình AI (LSTM + XGBoost), đến triển khai Backend FastAPI và giao diện Web Dashboard trực quan kèm cảnh báo cấp gió Beaufort cho khu vực Hải Phòng.